# Homework 5: Translation task (German → English)

[Qwen2.5-VL](https://huggingface.co/Qwen/Qwen2.5-VL-7B-Instruct) is a
vision-language model, but it is also a strong **text** chat model. Here we use
it **text-only** (no images) to translate the German transcript from
`00_PrepData` into English.

We translate **segment by segment** so each request stays short and the
timestamps are preserved — this also scales to clips of any length.

**Sources**
- Qwen2.5-VL: <https://huggingface.co/Qwen/Qwen2.5-VL-7B-Instruct> ([report](https://arxiv.org/abs/2502.13923))
- Model class `QwenChat` lives in `src/video_pipeline/qwen_text.py`.

In [1]:
import json
import shutil
import subprocess
import sys
from pathlib import Path

# Make the vendored `video_pipeline` package importable even if the project
# was not installed (e.g. running `jupyter` outside `uv run`).
SRC = Path.cwd() / "src"
if SRC.exists() and str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

PREPARED = Path("prepared")
PREPARED.mkdir(exist_ok=True)

transcript = json.loads((PREPARED / "transcript.json").read_text(encoding="utf-8"))
print(f"loaded transcript: {len(transcript['segments'])} segments, "
      f"{len(transcript['text'])} chars")
print("\nfirst 300 chars (German):\n", transcript["text"][:300])

loaded transcript: 9 segments, 1698 chars

first 300 chars (German):
 Das italienische Universal-Genie Leonardo da Vinci hat nicht nur die Mona Lisa und andere schöne Bilder gemalt, er war auch ein genialer Ingenieur. Viele Skizzen und Beschreibungen seiner Erfindungen sind bis heute erhalten. Zahnräder spielten bei seinen Erfindungen eine große Rolle. Das Zahnrad an 


## Translate each segment with Qwen2.5-VL

`QwenChat.translate(text, source, target)` wraps a single system+user chat turn.
We load the model once (`with QwenChat() as qc:`) and reuse it across all
segments, then it is unloaded and the GPU cache is cleared on exit.

In [2]:
from video_pipeline.qwen_text import QwenChat

# TODO (Exercise 2a): translate every segment from German into English.
translated_segments = []
with QwenChat() as qc:
    for i, seg in enumerate(transcript["segments"], start=1):
        english = qc.translate(seg["text"], source="german", target="english")
        translated_segments.append({**seg, "text": english})
        print(f"[{i}/{len(transcript['segments'])}] {english[:70]}")

english_text = "\n".join(s["text"] for s in translated_segments)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

/home/azureuser/Projects/GAI4_course/HW_5/.venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/home/azureuser/Projects/GAI4_course/HW_5/.venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[1/9] The Italian universal genius Leonardo da Vinci not only painted the Mo
[2/9] Or a heavy-duty forklift. Slightly modified, this corresponds to today
[3/9] He developed many weapons on behalf of his employers.
[4/9] For example, this organ gun. Quasi the first machine gun.
[5/9] And even a tank that, however, due to a construction defect could not 
[6/9] Bitte übersetzen Sie das deutsche Text in den englischen. Erhalten Sie
[7/9] Subtitling for ZDF for funk, 2017. Quasi the first machine gun. And ev
[8/9] His design of a parachute. A success, albeit only after 500 years.
[9/9] The British Adrian Nicholas sailed through the air in 2000.


Performance of the smaller model: 
* Peak Memory-Usage: 2,600 MiB
* Execution took 39.4s (when the model is already downloaded).

Performance of the bigger model: 
* Peak Memory-Usage: 6,040 MiB
* Execution took 3m 57.7s (when the model is already downloaded).

## Save a `TranslationResult`

We store the full English text in the project's `TranslationResult` schema so
the next notebook (and the CLI) can consume it.

In [ ]:
from video_pipeline.models import TranslationResult

result = TranslationResult(
    source_path=str(PREPARED / "transcript_smaller.json"),
    source_language="German",
    target_language="English",
    source_text=transcript["text"],
    text=english_text,
)
out = PREPARED / "translation.json"
out.write_text(result.model_dump_json(indent=2), encoding="utf-8")
print("wrote", out, "-", len(english_text), "chars")

wrote prepared/translation_smaller.json - 1666 chars


In [4]:
# German vs English, side by side per segment.
for de, en in zip(transcript["segments"][:6], translated_segments[:6]):
    print(f"[{de['start']:6.1f}s] DE: {de['text']}")
    print(f"          EN: {en['text']}\n")

[   1.0s] DE: Das italienische Universal-Genie Leonardo da Vinci hat nicht nur die Mona Lisa und andere schöne Bilder gemalt, er war auch ein genialer Ingenieur. Viele Skizzen und Beschreibungen seiner Erfindungen sind bis heute erhalten. Zahnräder spielten bei seinen Erfindungen eine große Rolle. Das Zahnrad an sich existierte bereits, aber da Vinci entwickelte daraus zahlreiche Maschinen. Zum Beispiel das Kettenradgetriebe, das wir heute in jedem Fahrrad finden.
          EN: The Italian universal genius Leonardo da Vinci not only painted the Mona Lisa and other beautiful pictures, he was also a brilliant engineer. Many of his sketches and descriptions of his inventions have survived to this day. Gears played a significant role in his inventions. The gear itself already existed, but Da Vinci developed numerous machines from it. For example, the chain drive mechanism, which we find today in every bicycle.

[  28.2s] DE: Oder einen Schwerlastheber. Etwas abgewandelt entspricht das dem 

As the transcription was way worse than with the bigger model, the translation results of the smaller model are also relatively poor:

```[   0.0s] DE: Das italienische Universalgenie Leonado Davinci hat nicht nur die Mona Lisa und andere schöne Bilder gemalt. Er war auch ein genialer Ingenieur. Vieles gibt es und beschreibungen seine Erfindungen sind bis heute erhalten. Zahnräder spielten bei seinen Erfindungen eine große Rolle. Das Zahnrad an sich existierte bereits, aber Davinci entwickelte daraus zahlreiche Maschinen. Zum Beispiel das Kettenradgetriebe, das wir heute in jedem Fahrrad finden. Oder einen schwerlastheber. Etwas abgewandelt entspricht das dem heutigen Wagenheber. Viele waffen entwickelte er im Auftrag seiner Arbeitgeber. Zum Beispiel dieses Orgelgeschütz. Quasi das erste Maschinengewehr. Und sogar einem Panzer, der sich allerdings wegen eines Konstruktionsfehlers nicht vorwärts bewegen konnte.
          EN: The Italian genius Leonardo da Vinci not only painted the Mona Lisa and other beautiful pictures. He was also a brilliant engineer. Many of his inventions and descriptions have been preserved to this day. Gears played a significant role in his inventions. The gear itself existed, but da Vinci developed numerous machines from it. For example, the chain drive that we find in every bicycle today. Or a heavy-lift hoist. Something modified corresponds to the current car jack. Many weapons he developed at the request of his employers. For example, this organ cannon. Almost like the first machine gun. And even a tank, which however could not move forward due to a design flaw.

[  50.0s] DE: Da winnschigend außerdem als Vater der Bionik.
          EN: Heinrich Hertz is also considered as the father of biophysics.

[  53.0s] DE: Für die Zeichnungen seiner Fluggeräte beobachtet der Air wie Füge ihre Flügel bewegen.
          EN: For his drawings of his aircraft, the Air watches Füge move its wings.

[  58.0s] DE: Danach entwarwer ein Fluggeräht mit Flügeln, die der Mensch aus eigener Kraft bewegen soll.
          EN: After that, he designed an aircraft with wings that should be moved by human effort.

[  63.0s] DE: Einen Orn die Topter, der blieb aber reine von Tasi. Kein Mensch wäre stark genug, um damit abzuhägen. Mit der Luftschraube sollten sich vier Personen senkrecht in die Luft drehen und fliegen. Ein Vorläufer des Hubschraubers, doch auch dieses Gerät war zu schwer. Es gab noch keine leichten Materialien, die Kunststoff oder Aluminium.
          EN: A flying machine that could fly but was not made of Tasi. No one would be strong enough to cut it down. With the propeller, four people should be able to rotate vertically and fly. A predecessor of the helicopter, but this device was too heavy. There were still no light materials, no plastics or aluminum.

[  87.0s] DE: Den Traum vom Fliegen gab da Wienchen nicht auf.
          EN: The dream of flying did not give up to the girl.

---
**Next:** `02_Summarize_OCR.ipynb` — summarise the English transcript and OCR the
on-screen text from the video frames.